# AI Hub Emotion TTS Fine-tuning

Run this notebook in the Google account that owns Colab Pro. The Codex app Google account is not used for Drive storage or training outputs.

Target pipeline: AI Hub emotional TTS data -> Fish Speech VQ tokens -> protobuf dataset -> `openaudio-s1-mini` LoRA fine-tuning -> emotion-conditioned inference.

## 0. Runtime

Use `Runtime > Change runtime type > GPU`. L4 or A100 is preferred. T4 can run smoke tests and small fine-tuning runs with reduced batch sizes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/gyul-ai/emotion-tts')
RAW_DIR = DRIVE_ROOT / 'raw'
PROCESSED_DIR = DRIVE_ROOT / 'processed'
PROTO_DIR = DRIVE_ROOT / 'protos'
CHECKPOINT_ROOT = DRIVE_ROOT / 'checkpoints'
RESULTS_DIR = DRIVE_ROOT / 'results'
SAMPLES_DIR = DRIVE_ROOT / 'generated_samples'

for path in [RAW_DIR, PROCESSED_DIR, PROTO_DIR, CHECKPOINT_ROOT, RESULTS_DIR, SAMPLES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)

## 1. Prepare AI Hub data

Download AI Hub `감정 음성합성 데이터셋` (`dataSetSn=286`) with your team account, then extract it under:

```text
/content/drive/MyDrive/gyul-ai/emotion-tts/raw
```

The repository never stores the AI Hub raw data.

In [ ]:
REPO_URL = 'https://github.com/novvvv/Gyul-AI-Repository.git'
BRANCH = 'MaTuna/tts'

%cd /content
!rm -rf Gyul-AI-Repository
!git clone {REPO_URL}
%cd /content/Gyul-AI-Repository
!git checkout {BRANCH}
!git status --short --branch

In [ ]:
%cd /content
!apt-get update -y
!apt-get install -y portaudio19-dev libasound2-dev ffmpeg libsox-dev libsndfile1
!rm -rf fish-speech
!git clone https://github.com/fishaudio/fish-speech.git fish-speech
%cd /content/fish-speech
!python -m pip install --upgrade pip
!python -m pip install -U uv 'huggingface_hub>=0.34.0,<1.0'
!uv pip install --system -e '.[cu126]'
!python -m pip uninstall -y torchvision

In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print('vram_gb:', round(total_gb, 2))

## Hugging Face access

`fishaudio/openaudio-s1-mini` is gated. Before running the checkpoint download cell, open the model page while logged into Hugging Face, accept/request access, then log in from this notebook with a read token.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()
!hf auth whoami

In [ ]:
BASE_CKPT = CHECKPOINT_ROOT / 'openaudio-s1-mini'
!hf download fishaudio/openaudio-s1-mini --local-dir "{BASE_CKPT}" --max-workers 1
print('Base checkpoint:', BASE_CKPT)

In [ ]:
%cd /content/Gyul-AI-Repository
!python scripts/prepare_aihub_emotion_dataset.py \
  --input-dir "{RAW_DIR}" \
  --output-dir "{PROCESSED_DIR}" \
  --samples-per-emotion 300 \
  --overwrite

!find "{PROCESSED_DIR}" -maxdepth 2 -type f | head -20

For smoke tests, change `--samples-per-emotion 300` above to `2` or `10` before running VQ extraction.

In [ ]:
%cd /content/fish-speech

# T4: use 4. L4/A100: use 16.
VQ_BATCH_SIZE = 4

!python tools/vqgan/extract_vq.py "{PROCESSED_DIR}" \
  --num-workers 1 \
  --batch-size {VQ_BATCH_SIZE} \
  --config-name modded_dac_vq \
  --checkpoint-path "{BASE_CKPT / 'codec.pth'}"

In [ ]:
%cd /content/fish-speech
!rm -rf "{PROTO_DIR}"
!python tools/llama/build_dataset.py \
  --input "{PROCESSED_DIR}" \
  --output "{PROTO_DIR}" \
  --text-extension .lab \
  --num-workers 4

!find "{PROTO_DIR}" -maxdepth 1 -type f -name '*.protos' -print

In [ ]:
%cd /content/fish-speech
!rm -rf data/protos
!mkdir -p data
!ln -s "{PROTO_DIR}" data/protos
!ls -la data/protos

In [ ]:
%cd /content/fish-speech

PROJECT_NAME = 'aihub_emotion_lora'
MAX_STEPS = 1500
BATCH_SIZE = 2
GRAD_ACCUM = 8
PRECISION = 'bf16-true'  # if T4 fails, try '16-mixed'

!python fish_speech/train.py --config-name text2semantic_finetune \
  project={PROJECT_NAME} \
  pretrained_ckpt_path="{BASE_CKPT}" \
  data.batch_size={BATCH_SIZE} \
  trainer.accumulate_grad_batches={GRAD_ACCUM} \
  trainer.max_steps={MAX_STEPS} \
  trainer.precision={PRECISION} \
  hydra.run.dir="{RESULTS_DIR / PROJECT_NAME}" \
  +lora@model.model.lora_config=r_8_alpha_16

## 2. Merge LoRA

Set `LORA_CKPT` to the checkpoint produced by the previous cell. For example:

```text
/content/drive/MyDrive/gyul-ai/emotion-tts/results/aihub_emotion_lora/checkpoints/step_000000100.ckpt
```

In [ ]:
%cd /content/fish-speech

LORA_CKPT = ''  # TODO: paste the selected .ckpt path after training
MERGED_CKPT = CHECKPOINT_ROOT / 'openaudio-s1-mini-aihub-emotion-lora'

assert LORA_CKPT, 'Set LORA_CKPT before running this cell.'
!python tools/llama/merge_lora.py \
  --lora-config r_8_alpha_16 \
  --base-weight "{BASE_CKPT}" \
  --lora-weight "{LORA_CKPT}" \
  --output "{MERGED_CKPT}"

print('Merged checkpoint:', MERGED_CKPT)

In [ ]:
from pathlib import Path

reference_audio = next(PROCESSED_DIR.rglob('*.wav'))
reference_text = reference_audio.with_suffix('.lab')

print('reference_audio:', reference_audio)
print('reference_text:', reference_text)

In [ ]:
%cd /content/fish-speech

import shutil
import subprocess

sample_text = "(sad) 괜찮아요. 천천히 이야기해도 돼요."
prompt_text = Path(reference_text).read_text(encoding="utf-8").strip()
codec_path = BASE_CKPT / "codec.pth"
sample_output = SAMPLES_DIR / "sad_sample.wav"

subprocess.run([
    "python", "fish_speech/models/dac/inference.py",
    "-i", str(reference_audio),
    "--checkpoint-path", str(codec_path),
], check=True)
subprocess.run([
    "python", "fish_speech/models/text2semantic/inference.py",
    "--text", sample_text,
    "--prompt-text", prompt_text,
    "--prompt-tokens", "fake.npy",
    "--checkpoint-path", str(MERGED_CKPT),
    "--half",
], check=True)
codes_path = Path("output/codes_0.npy") if Path("output/codes_0.npy").exists() else Path("codes_0.npy")
subprocess.run([
    "python", "fish_speech/models/dac/inference.py",
    "-i", str(codes_path),
    "--checkpoint-path", str(codec_path),
], check=True)
shutil.copy2("fake.wav", sample_output)
print("Saved sample:", sample_output)

In [ ]:
from IPython.display import Audio
Audio(str(SAMPLES_DIR / 'sad_sample.wav'))